# Checkpoint 10: How a model learns from features

**Goal:** understand logistic regression before using it on shipment data. Run independently with `.venv`.

Checkpoint 9 showed that a constant comparison model cannot distinguish shipments. Our next candidate should use the features to produce different probabilities. Logistic regression is a first candidate for learning and later evaluation, not a declared winner.

Recall matters, but always alerting would catch every incident while creating many false alarms. We also need precision (how many alerts correspond to actual incidents), probability quality, and valid future-data evaluation. No single number selects the model.

This notebook fits a small logistic model on **invented teaching examples only**. They deliberately contain a simple relationship so we can see the learning process. They are not evidence about refrigerated-shipment safety or actual model performance. We do not load or train on the real shipment dataset here.


## 1. A weight gives a feature influence

With one feature, the model first computes a score:

**score = intercept + weight × feature**

- Feature: input measured for an example.
- Weight: learned strength/direction of that feature's contribution to the score.
- Intercept: learned starting score before the feature contribution.

A positive weight makes a higher feature increase the score; a negative weight decreases it. The score is not yet a probability and can be negative or greater than one.

The logistic function (also called sigmoid) converts the score to a probability between zero and one. Logistic regression is a classification method despite its name. With multiple features, it adds their weighted contributions before this conversion. A coefficient is not automatically causal importance, especially with correlated features.


In [1]:
import math
from statistics import mean, pstdev
import pandas as pd
from IPython.display import display

def sigmoid(score):
    if score >= 0:
        return 1 / (1 + math.exp(-score))
    e = math.exp(score)
    return e / (1 + e)

display(pd.DataFrame([{"score": z, "probability": sigmoid(z)} for z in (-2, -1, 0, 1, 2)]))
assert sigmoid(0) == 0.5
assert 0 < sigmoid(-2) < sigmoid(2) < 1


,score,probability
0,-2,0.119203
1,-1,0.268941
2,0,0.500000
3,1,0.731059
4,2,0.880797


## 2. Teaching examples and scaling

Our invented examples pair a temperature input with a made-up binary outcome. Their values do not define real safe-temperature limits.

For this exercise we subtract the training mean and divide by the training standard deviation. This is **feature scaling**, different from flattening JSON into tables. It changes units, not the information available. Save these training statistics and reuse them for new inputs; never fit them on evaluation data.

Missing values are absent from this toy example. The real table does contain missing slopes, so its later preparing model inputs must explicitly handle those using training-only fitted statistics and indicators.


In [2]:
toy_temperatures = [2., 3., 4., 5., 6., 7., 8., 9.]
toy_labels = [0, 0, 0, 1, 0, 1, 1, 1]
training_mean = mean(toy_temperatures)
training_scale = pstdev(toy_temperatures)
x = [(value - training_mean) / training_scale for value in toy_temperatures]
y = toy_labels
display(pd.DataFrame({"invented_temperature_c": toy_temperatures,
                      "scaled_feature": x, "invented_label": y}))
assert abs(mean(x)) < 1e-12


,invented_temperature_c,scaled_feature,invented_label
0,2.0,-1.527525,0
1,3.0,-1.091089,0
2,4.0,-0.654654,0
3,5.0,-0.218218,1
4,6.0,0.218218,0
5,7.0,0.654654,1
6,8.0,1.091089,1
7,9.0,1.527525,1


## 3. What training actually does

Begin with weight and intercept at zero. Every score is zero, so every probability is 50%. This is an initialization for the toy experiment, not a claim that our real incident rate is 50%.

Training repeatedly:
1. Calculates probabilities from the current weight and intercept.
2. Compares probabilities with labels using **log loss** (an error measure that strongly penalizes confidently wrong predictions).
3. Adjusts the parameters a little in a direction that reduces the objective.

The adjustment method here is **gradient descent**. Learning rate controls the step size. We add a small squared-weight penalty, called L2 regularization, to discourage unnecessarily large weights. The regularization setting is chosen for teaching, not tuned on shipment outcomes. Library solvers may use a different optimization method for the same model family.

The formulas in the next cell expose the mechanics. You do not need to memorize derivatives: understand predict → compare → adjust → repeat. Lower training loss does not establish generalization.


In [3]:
def objective(weight, intercept, xs, ys, penalty):
    losses = []
    for value, label in zip(xs, ys):
        z = intercept + weight * value
        # Stable binary log loss: log(1 + exp(z)) - label*z.
        losses.append(max(z, 0) - label*z + math.log1p(math.exp(-abs(z))))
    return mean(losses) + 0.5 * penalty * weight * weight

def fit_toy(xs, ys, steps=1000, learning_rate=0.1, penalty=0.05):
    weight, intercept = 0.0, 0.0
    history = []
    for step in range(steps + 1):
        if step in (0, 1, 10, 100, steps):
            history.append({"step": step, "weight": weight, "intercept": intercept,
                            "training_objective": objective(weight, intercept, xs, ys, penalty)})
        if step == steps:
            break
        errors = [sigmoid(intercept + weight*value) - label for value, label in zip(xs, ys)]
        weight_gradient = mean(error*value for error, value in zip(errors, xs)) + penalty*weight
        intercept_gradient = mean(errors)
        weight -= learning_rate * weight_gradient
        intercept -= learning_rate * intercept_gradient
    return weight, intercept, history

weight, intercept, history = fit_toy(x, y)
display(pd.DataFrame(history))
assert history[-1]["training_objective"] < history[0]["training_objective"]
assert fit_toy(x, y) == (weight, intercept, history)
# A finite-difference check independently verifies the weight gradient at one point.
w, b, penalty, epsilon = 0.3, -0.2, 0.05, 1e-6
analytic = mean((sigmoid(b+w*v)-label)*v for v,label in zip(x,y)) + penalty*w
numeric = (objective(w+epsilon,b,x,y,penalty)-objective(w-epsilon,b,x,y,penalty))/(2*epsilon)
assert abs(analytic-numeric) < 1e-7
print("Objective decreased; repeat fitting is deterministic; gradient check passed.")


,step,weight,intercept,training_objective
0,0,0.000000,0.000000e+00,0.693147
1,1,0.038188,0.000000e+00,0.678783
2,10,0.334535,-7.632783e-18,0.582069
3,100,1.370428,-1.127570e-17,0.425530
4,1000,1.610077,-2.046974e-17,0.420919


Objective decreased; repeat fitting is deterministic; gradient check passed.


## 4. Use the learned model without providing new labels

The new input is scaled using the original training mean and standard deviation, then passed through the learned score and probability mapping. These are toy predictions. None is a validated operational risk estimate.


In [4]:
def toy_predict(temperature):
    scaled = (temperature - training_mean) / training_scale
    return sigmoid(intercept + weight * scaled)

predictions = pd.DataFrame({"new_temperature_c": [3.5, 5.5, 7.5],
                            "toy_probability": [toy_predict(t) for t in [3.5, 5.5, 7.5]]})
display(predictions)
assert all(0 < value < 1 for value in predictions["toy_probability"])
print("Prediction used saved scaling and learned parameters; it did not receive new labels.")


,new_temperature_c,toy_probability
0,3.5,0.196962
1,5.5,0.500000
2,7.5,0.803038


Prediction used saved scaling and learned parameters; it did not receive new labels.


## 5. How is a decision tree different?

| Question | Logistic regression | Decision tree |
|---|---|---|
| How does it use features? | Adds weighted feature contributions, then converts score to probability | Learns branching questions such as whether a feature exceeds a threshold |
| What gets learned? | Weights and intercept | Split features, thresholds, and leaf outcome frequencies |
| Where do probabilities come from? | Logistic mapping of the learned score | Class proportions in the reached leaf, for a standard classification tree |
| Useful quality | Compact, inspectable starting model | Can capture thresholds and feature interactions |
| Important limitation | May need engineered transformations/interactions | A deep tree can memorize noise; small leaves can give unreliable probabilities |

A tree's learned split is not automatically a scientifically established safety threshold. Logistic probabilities are not automatically calibrated either. Both need future-data evaluation. We have not fitted a tree or compared candidate models here.

**Working approach:** retain the constant comparison model, evaluate a regularized logistic regression first, and consider a shallow tree as a modest nonlinear comparison. This is a sequence of experiments, not a preselected winner. More complex ensembles are optional if justified by evidence and the assignment timebox.

Before real fitting: define ordered by time train/validation/test windows, label availability and maturity at each fit cutoff, final test outcome coverage, and shipment overlap handling. Fit missing-value processing and scaling only on training data. Tune decisions on validation, reserve final test outcomes for evaluation.

Sources: [scikit-learn logistic regression guide](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression), [scikit-learn decision-tree guide](https://scikit-learn.org/stable/modules/tree.html#classification).


## 6. Check your understanding

A model learns parameters from examples; feature engineering defines its inputs. Training loss helps adjust those parameters. Evaluation on later unseen examples determines whether it learned a useful pattern rather than merely fitting the examples.

**Interview notes:** “I started with a constant comparison model, then considered regularized logistic regression as an inspectable model that can use shipment features. A shallow tree provides a nonlinear comparison. Selection depends on time-valid ranking and probability metrics, not training accuracy.”

**Try explaining this:** what changes during logistic-regression training. the original temperature readings, or the model's weights and intercept?

**Next:** establish the real ordered by time evaluation and preparing model inputs in notebook 11, then fit and compare the baseline and candidate models. Toy results here must not appear as shipment evaluation metrics.
